In [37]:
import pandas as pd
!pip install geopy

  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached geographiclib-2.1-py3-none-any.whl.metadata (1.6 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
Using cached geographiclib-2.1-py3-none-any.whl (40 kB)


In [2]:
df = pd.read_excel("intercity_bus_dataset_whitefield.xlsx")
df.head()

,Bus_No,Operator,Route,Stops,Departure,Duration_hrs
0,BUS200,Sugama,Mangalore-Bangalore,Jyothi Lalbagh Udupi Kundapura Bhatkal Honnava...,05:30 AM,11
1,BUS201,KSRTC,Mangalore-Bangalore,Statebank Pumpwell Udupi Kundapura Byndoor Bha...,09:00 AM,8
2,BUS202,Sugama,Mangalore-Bangalore,Jyothi Kottara Surathkal Udupi Kundapura Bhatk...,10:00 AM,8
3,BUS203,Durgamba,Mangalore-Bangalore,Jyothi Lalbagh Udupi Kundapura Bhatkal Honnava...,07:00 PM,8
4,BUS204,KSRTC,Mangalore-Bangalore,Jyothi Lalbagh Udupi Kundapura Bhatkal Honnava...,06:30 AM,8


In [3]:
df["Stops"] = df["Stops"].str.lower()

In [26]:
nearest_map = {
    "pandeshwar": ["jyothi", "lalbagh"],
    "kankanady": ["lalbagh", "jyothi"],
    "pumpwell": ["statebank"],
    "btm": ["silk board"],
    "indiranagar": ["marathahalli", "majestic"],
    
    # 🔥 ADD THESE (IMPORTANT)
    "atavar": ["jyothi", "lalbagh"],
    "falnir": ["jyothi"],
    "hampankatta": ["jyothi"]
}

In [27]:
def get_nearest_stop(user_location, stops):
    stops_list = stops.split()
    
    # direct match
    if user_location.lower() in stops_list:
        return user_location.lower()
    
    # check mapped nearby stops
    if user_location.lower() in nearest_map:
        for stop in nearest_map[user_location.lower()]:
            if stop in stops_list:
                return stop
    
    return None

In [28]:
def recommend_parcel(source, destination):
    
    results = []
    
    for i, row in df.iterrows():
        stops = row["Stops"]
        stops_list = stops.split()
        
        pickup = get_nearest_stop(source, stops)
        drop = get_nearest_stop(destination, stops)
        
        if pickup and drop:
            if stops_list.index(pickup) < stops_list.index(drop):
                
                results.append({
                    "Bus": row["Bus_No"],
                    "Operator": row["Operator"],
                    "Pickup": pickup,
                    "Drop": drop,
                    "Departure": row["Departure"],
                    "Duration": row["Duration_hrs"]
                })
    
    return results

In [41]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="parcel_app")

def get_coordinates(place):
    location = geolocator.geocode(place)
    if location:
        return (location.latitude, location.longitude)
    return None

In [42]:
from geopy.distance import geodesic

def get_distance(user_loc, pickup_loc):
    return geodesic(user_loc, pickup_loc).km

In [48]:
import difflib
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

# ------------------ GEO SETUP ------------------
geolocator = Nominatim(user_agent="parcel_app")

def get_coordinates(place):
    try:
        location = geolocator.geocode(place)
        if location:
            return (location.latitude, location.longitude)
    except:
        return None
    return None

def get_distance(loc1, loc2):
    if loc1 and loc2:
        return round(geodesic(loc1, loc2).km, 2)
    return None


# ------------------ GET ALL STOPS ------------------
all_stops = set()
for stops in df["Stops"]:
    for s in stops.split():
        all_stops.add(s)

all_stops = list(all_stops)


# ------------------ SPELL CORRECTION ------------------
def correct_spelling(user_input):
    match = difflib.get_close_matches(user_input.lower(), all_stops, n=1, cutoff=0.6)
    return match[0] if match else user_input.lower()


# ------------------ SMART DROP ------------------
def get_best_drop(destination, stops):
    stops_list = stops.split()
    
    if destination.lower() in stops_list:
        return destination.lower()
    
    if destination.lower() in nearest_map:
        for stop in nearest_map[destination.lower()]:
            if stop in stops_list:
                return stop
    
    match = difflib.get_close_matches(destination.lower(), stops_list, n=1, cutoff=0.4)
    if match:
        return match[0]
    
    return None


# ------------------ FALLBACK VIA MAJOR ------------------
def recommend_via_major_stop(source, major_stop="majestic"):
    results = []
    
    for i, row in df.iterrows():
        stops_list = row["Stops"].split()
        pickup = get_nearest_stop(source, row["Stops"])
        
        if pickup and major_stop in stops_list:
            if stops_list.index(pickup) < stops_list.index(major_stop):
                
                results.append({
                    "Bus": row["Bus_No"],
                    "Operator": row["Operator"],
                    "Pickup": pickup,
                    "Drop": major_stop,
                    "Departure": row["Departure"],
                    "Duration": row["Duration_hrs"]
                })
    
    return results


# ------------------ CHAT FUNCTION ------------------
def chat_show_results():
    
    print("Smart Parcel Chat System\n")
    
    source = input("Enter Pickup Location: ").lower()
    destination = input("Enter Drop Location: ").lower()
    
    source = correct_spelling(source)
    destination = correct_spelling(destination)
    
    print(f"\nInterpreted as: {source.title()} -> {destination.title()}")
    
    direct_results = []
    nearest_results = []
    
    for i, row in df.iterrows():
        stops_list = row["Stops"].split()
        
        pickup = get_nearest_stop(source, row["Stops"])
        
        if not pickup:
            continue
        
        if destination.lower() in stops_list:
            drop = destination.lower()
            
            if stops_list.index(pickup) < stops_list.index(drop):
                direct_results.append({
                    "Bus": row["Bus_No"],
                    "Operator": row["Operator"],
                    "Pickup": pickup,
                    "Drop": drop,
                    "Departure": row["Departure"],
                    "Duration": row["Duration_hrs"]
                })
        
        else:
            drop = get_best_drop(destination, row["Stops"])
            
            if drop and stops_list.index(pickup) < stops_list.index(drop):
                nearest_results.append({
                    "Bus": row["Bus_No"],
                    "Operator": row["Operator"],
                    "Pickup": pickup,
                    "Drop": drop,
                    "Departure": row["Departure"],
                    "Duration": row["Duration_hrs"]
                })
    
    if direct_results:
        print("\nDirect Bus Routes Available:\n")
        results = direct_results
    
    elif nearest_results:
        print(f"\nNo direct bus to {destination.title()}")
        print("Showing nearest drop points:\n")
        results = nearest_results
    
    else:
        print(f"\nNo direct or nearby drop available for {destination.title()}")
        print("Showing routes via Majestic:\n")
        
        results = recommend_via_major_stop(source, "majestic")
        
        if not results:
            print("No route found at all.")
            return
    
    for r in results[:3]:
        
        print("Bus:", r["Bus"], "|", r["Operator"])
        print("Pickup Point:", r["Pickup"].title())
        print("Drop Point:", r["Drop"].title())
        print("Departure:", r["Departure"])
        print("Duration:", r["Duration"], "hours")
        
        user_loc = get_coordinates(source + " Mangalore")
        pickup_loc = get_coordinates(r["Pickup"] + " Mangalore")
        
        dist = get_distance(user_loc, pickup_loc)
        
        if dist:
            print(f"Distance to Pickup: {dist} km")
        else:
            print("Distance: Not available")
        
        route_row = df[df["Bus_No"] == r["Bus"]].iloc[0]
        print("Full Route:", route_row["Stops"].title())
        
        stops_list = route_row["Stops"].split()
        start = stops_list.index(r["Pickup"])
        end = stops_list.index(r["Drop"])
        
        sub_route = stops_list[start:end+1]
        print("Route (Pickup -> Drop):", " -> ".join(sub_route).title())
        
        print("------")
    
    if not direct_results:
        print("\nFinal Step:")
        print(f"Use local transport from {results[0]['Drop'].title()} to {destination.title()}")


# ------------------ RUN ------------------
chat_show_results()

Smart Parcel Chat System



Enter Pickup Location:  atavar
Enter Drop Location:  nelmangala



Interpreted as: Atavar -> Nelamangala

Direct Bus Routes Available:

Bus: BUS200 | Sugama
Pickup Point: Jyothi
Drop Point: Nelamangala
Departure: 05:30 AM
Duration: 11 hours
Distance: Not available
Full Route: Jyothi Lalbagh Udupi Kundapura Bhatkal Honnavar Hassan Nelamangala Majestic Silk Board Marathahalli Whitefield
Route (Pickup -> Drop): Jyothi -> Lalbagh -> Udupi -> Kundapura -> Bhatkal -> Honnavar -> Hassan -> Nelamangala
------
Bus: BUS203 | Durgamba
Pickup Point: Jyothi
Drop Point: Nelamangala
Departure: 07:00 PM
Duration: 8 hours
Distance: Not available
Full Route: Jyothi Lalbagh Udupi Kundapura Bhatkal Honnavar Hassan Nelamangala Majestic Silk Board Marathahalli Whitefield
Route (Pickup -> Drop): Jyothi -> Lalbagh -> Udupi -> Kundapura -> Bhatkal -> Honnavar -> Hassan -> Nelamangala
------
Bus: BUS204 | KSRTC
Pickup Point: Jyothi
Drop Point: Nelamangala
Departure: 06:30 AM
Duration: 8 hours
Distance: Not available
Full Route: Jyothi Lalbagh Udupi Kundapura Bhatkal Honnavar 